In [2]:
text = """
Área: GGMON

Número: 6

Ano: 2026
Resumo:

Descontinuidade temporária da estratégia de vacinação com a vacina BUTANTAN-DV contra dengue (tetravalente, atenuada, dose única) como medida de precaução, enquanto é investigada a ocorrência de eventos graves (raros) em pessoas vacinadas.

Identificação do produto ou caso:

Vacina Butantan-DV (vacina dengue 1,2,3 e 4 atenuada) registrada pelo Instituto Butantan.

Problema:

Em dezembro de 2025, a vacina BUTANTAN-DV foi registrada pela Anvisa para comercialização no Brasil. Em 17 de janeiro de 2026, o Programa Nacional de Imunizações (PNI), do Ministério da Saúde (MS), iniciou a estratégia nacional de vacinação com essa vacina, inicialmente como projeto piloto nos municípios de Botucatu (SP), Nova Lima (MG) e Maranguape (CE). Posteriormente, a partir de 9 de fevereiro de 2026, a vacinação foi expandida para a Atenção Primária à Saúde (APS) em todo o território nacional, tendo como população-alvo indivíduos de 15 a 59 anos. Até o dia 30 de maio, haviam sido administradas mais de 500 mil doses da vacina.

Após o início da vacinação, foram detectados 42 casos semelhantes a dengue com sinais de alarme e três casos semelhantes a dengue grave entre as notificações de Eventos Supostamente Atribuíveis à Vacinação ou Imunização (ESAVI), sendo que destes, dois casos evoluíram para óbito. Esses sinais de alarme não haviam sido observados durante os estudos clínicos realizados antes do registro da vacina.

Ainda não é possível afirmar a associação da vacina como a causa de todos os casos de dengue grave e dengue com sinais de alarme. No entanto, diante da gravidade do sinal de segurança detectado, e seguindo o princípio da precaução, a Anvisa e o Ministério da Saúde decidiram pela descontinuação temporária da vacinação com a BUTANTAN-DV até que as investigações sejam concluídas.


Ação:

Foi determinada pelo Ministério da Saúde a descontinuação temporária de vacinação com a vacina BUTANTAN-DV com o objetivo de cessar a exposição à vacina, proteger a população e garantir que todas as informações necessárias sejam avaliadas antes que novas decisões sobre o uso da vacina sejam tomadas.

A Anvisa determinou as seguintes medidas ao Instituto Butantan:

Monitoramento das pessoas vacinadas recentemente;
Notificação, no sistema Vigimed, de todos os casos de ESAVI graves do seu sistema de farmacovigilância com a avaliação feita pela empresa;
Análise e investigação dos casos que apresentaram ESAVI compatíveis com dengue com sinais de alarme e dengue grave; e
Atualização do Plano de Gerenciamento de Risco.



Para apoiar a Anvisa nas exigências e nas análises dos dados e das informações produzidas durante e após as investigações do sinal de segurança, a Anvisa criará um grupo de trabalho específico para, em conjunto com um painel de especialistas, aprofundar a investigação do sinal de segurança detectado. O objetivo é ampliar as análises e reunir evidências que auxiliem a Agência na reavaliação do perfil de segurança da vacina.


Histórico:

 Não houve alertas anteriores sobre o tema.


Recomendações:

Recomendações para os profissionais de saúde:

Observar atentamente as orientações contidas na Nota Técnica n 58/2026-DPNI/SVSA/MS

Suspender a utilização da vacina durante o período de descontinuação da estratégia de vacinação

Recomenda-se que as salas de vacinação e os serviços de atenção primária mantenham lista dos vacinados nos últimos 21 dias, quando disponível, para apoiar a comunicação ativa, orientação clínica e a busca de atendimento em caso de sintomas.

Orientar pessoas previamente vacinadas que apresentem, até 21 dias após a vacinação, sintomas compatíveis com dengue ou reação semelhante a dengue, incluindo febre e, pelo menos, mais dois seguintes sinais e sintomas: cefaleia, dor retro-orbital ou dor ocular, mialgia, artralgia, náuseas, vômitos, exantema, petéquias ou prova do laço positiva e leucopenia.

Fique alerta para os sinais de agravamento: dor abdominal intensa e contínua, vômitos persistentes, queda de pressão ao levantar ou desmaio, sangramento em mucosas, sonolência excessiva ou irritabilidade, fígado aumentado, acúmulo de líquido no corpo, aumento progressivo do hematócrito, piora clínica súbita, sinais de desidratação ou instabilidade hemodinâmica.

Considere dengue grave quando houver: choque ou queda persistente de pressão, sangramento grave, comprometimento de órgãos vitais (fígado, rins, pulmões, coração), alterações neurológicas, necessidade de UTI ou risco de óbito.

Para o profissional de saúde, a notificação de ESAVI deve ser realizada no sistema online e-SUS Notifica (https://notifica.saude.gov.br/), sendo compulsória e imediata a notificação de casos graves.

A população em geral pode realizar a notificação espontânea de eventos adversos no sistema VigiMed (https://vigiflow-eforms.who-umc.org/br/vigimed)



Recomendações para o público em geral:

Pessoas vacinadas continuam potencialmente protegidas contra os quatro tipos de dengue.

Para quem já recebeu a vacina, fique atento aos sinais e sintomas pelos 21 dias seguintes à vacinação. Procure atendimento imediatamente se apresentar febre, dor abdominal intensa, vômitos persistentes, sangramentos, tontura, sonolência excessiva, sinais de desidratação ou piora do estado geral.
"""

In [1]:
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device='cuda')

Device set to use cuda


In [3]:
def prompt(texto: str):
    return f"""
Instruções:
-Preserve dia, mês e ano dos eventos.
-Data da suspenção/descontinuação/alerta/infração é a informação mais importante.
-Saída somente o texto resumido, sem as instruções, em português.

Texto:
{texto}
"""

In [5]:
CHUNK_SIZE = 512
PROMPT_SIZE = len(prompt(""))

tokens = summarizer.tokenizer.encode(text)
chunks = []

for i in range(0, len(tokens), CHUNK_SIZE):
    chunk_tokens = tokens[i:i+CHUNK_SIZE]
    chunks.append(
        summarizer.tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=False,
            truncation=True,
            clean_up_tokenization_spaces=True,
        )
    )

partial_summaries = [
    summarizer(
        prompt(chunk),
        max_new_tokens=CHUNK_SIZE + PROMPT_SIZE,
        min_length=30,
        do_sample=False,
        truncation=True,
    )[0]["summary_text"]
    for chunk in chunks
]

final_summary = summarizer(
    prompt("".join(partial_summaries)),
    max_new_tokens=1024,
    min_length=50,
    do_sample=False
)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Your max_length is set to 142, but your input_length is only 121. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=60)


In [6]:
final_summary

[{'summary_text': 'Vacina Butantan-DV (vacina dengue 1,2,3 and 4 atenuada) registrada pelo Instituto Butantans. Estratégia nacional de vacinação foi expandida para a Atenção Primária à Saúde. Até o dia 30 de maio, haviam sido administradas mais de 500 mil doses.'}]